# Anchored L2 × 5 Frozen Forecasters × 120 Conditions

This notebook evaluates **last-value-anchored L2 retrieval** with the same frozen-forecaster
integration protocol used for full PRR.

## Frozen benchmark

- datasets: ETTh1, Weather, Electricity, Traffic, Exchange, Solar
- horizons: 96, 192, 336, 720
- backbones: PatchTST, iTransformer, TimeMixer, Seg-MoE, DLinear
- 6 × 4 × 5 = **120 forecasting conditions**
- exact Exp33 frozen validation/test query-channel manifests
- exact frozen Direct prediction caches
- no forecaster training or inference
- no retrieval learning

## Anchored-L2 retrieval

For a past window \(x_{1:L}\),

\[
\tilde x_t = x_t - x_L.
\]

Candidates are ranked by squared Euclidean distance between endpoint-anchored past windows.
Retrieval searches the full temporally admissible same-channel memory with

- lookback \(L=96\),
- memory stride 8,
- Top-K = 10,
- candidate future fully observed before the validation/test memory boundary.

The retrieved forecast is the mean candidate **endpoint-relative future**, re-anchored at the
current query endpoint.

## Downstream trust calibration

For each condition \(c=(\text{dataset},\text{horizon},\text{backbone})\), select

\[
\beta_c^\star
=
\arg\min_{\beta\in\{0,.025,.05,.075,.10,.15,.20\}}
\operatorname{MSE}_{\mathrm{val},c}
\]

using validation only, then freeze that beta and evaluate once on test:

\[
\hat y = \hat y_D + \beta_c^\star(\hat y_{L2}-\hat y_D).
\]

The notebook also reports the common fixed-\(\beta=0.1\) control and, when available,
compares the final L2 integration directly with the existing calibrated PRR 120-condition result.

In [1]:
# Cell 1 — setup and robust path discovery

from pathlib import Path
import os, io, json, time, gc
import numpy as np
import pandas as pd

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)
BACKBONES = ("PatchTST","iTransformer","TimeMixer","SegMoE","DLinear")

L = 96
MEMORY_STRIDE = 8
TOP_K = 10

BETA_GRID = np.array([0.0,0.025,0.05,0.075,0.10,0.15,0.20], dtype=np.float64)
FIXED_BETA = 0.10
TOL = 1e-10

def discover_dir(relative):
    preferred = [
        Path("/data/code/which_histories_matter_ext") / relative,
        Path("/data/code/stock_regime_retrieval/strong_forecaster") / relative,
    ]
    for p in preferred:
        if p.exists():
            return p

    parts = Path(relative).parts
    target_name = parts[-1]

    for base in [Path("/data/code"), Path("/data")]:
        if not base.exists():
            continue
        for p in base.rglob(target_name):
            if p.is_dir() and str(p).endswith(str(relative)):
                return p
    return None

MANIFEST_ROOT = discover_dir(Path("results/exp33/final_eval_manifest"))
PATCHTST_DIRECT_ROOT = discover_dir(Path("results/exp34/patchtst_direct_cache"))
OTHER_DIRECT_ROOT = discover_dir(Path("results/exp37D/direct_cache"))

assert MANIFEST_ROOT is not None, "Exp33 final_eval_manifest not found"
assert PATCHTST_DIRECT_ROOT is not None, "Exp34 PatchTST direct cache not found"
assert OTHER_DIRECT_ROOT is not None, "Exp37D other-backbone direct cache not found"

PROJECT_ROOT = MANIFEST_ROOT.parents[2]

OUT_ROOT = PROJECT_ROOT / "results" / "anchored_l2_downstream_120"
RETRIEVAL_CACHE_ROOT = OUT_ROOT / "retrieval_cache"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RETRIEVAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("MANIFEST_ROOT       :", MANIFEST_ROOT)
print("PATCHTST_DIRECT_ROOT:", PATCHTST_DIRECT_ROOT)
print("OTHER_DIRECT_ROOT   :", OTHER_DIRECT_ROOT)
print("PROJECT_ROOT        :", PROJECT_ROOT)
print("OUT_ROOT            :", OUT_ROOT)
print("BETA_GRID           :", BETA_GRID.tolist())

MANIFEST_ROOT       : /data/code/which_histories_matter_ext/results/exp33/final_eval_manifest
PATCHTST_DIRECT_ROOT: /data/code/which_histories_matter_ext/results/exp34/patchtst_direct_cache
OTHER_DIRECT_ROOT   : /data/code/which_histories_matter_ext/results/exp37D/direct_cache
PROJECT_ROOT        : /data/code/which_histories_matter_ext
OUT_ROOT            : /data/code/which_histories_matter_ext/results/anchored_l2_downstream_120
BETA_GRID           : [0.0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2]


In [2]:
# Cell 2 — exact frozen manifest loader

def load_manifest(dataset, horizon, split):
    p = MANIFEST_ROOT / f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(), p

    with np.load(p, allow_pickle=False) as z:
        a = np.asarray(z["anchor"], np.int64)
        c = np.asarray(z["channel"], np.int64)

    assert a.shape == c.shape
    assert len(a) > 0
    return a, c

rows = []
for d in DATASETS:
    for h in HORIZONS:
        for sp in ("val","test"):
            a,c = load_manifest(d,h,sp)
            rows.append({
                "dataset":d,
                "horizon":h,
                "split":sp,
                "pairs":len(a),
                "channels":len(np.unique(c)),
                "anchor_min":int(a.min()),
                "anchor_max":int(a.max()),
            })

manifest_df = pd.DataFrame(rows)
display(manifest_df)
assert len(manifest_df) == 48
print("[PASS] frozen manifests 48/48")

,dataset,horizon,split,pairs,channels,anchor_min,anchor_max
0,ETTh1,96,val,4096,7,8640,11424
1,ETTh1,96,test,8192,7,11520,17323
2,ETTh1,192,val,4096,7,8640,11328
3,ETTh1,192,test,8192,7,11520,17228
4,ETTh1,336,val,4096,7,8640,11184
5,ETTh1,336,test,8192,7,11520,17084
6,ETTh1,720,val,4096,7,8640,10800
7,ETTh1,720,test,8192,7,11520,16700
8,Weather,96,val,4096,21,36887,42061
9,Weather,96,test,8192,21,42159,52600


[PASS] frozen manifests 48/48


In [3]:
# Cell 3 — exact Direct cache loader (copied from the frozen PRR calibration protocol)

def direct_slug(backbone):
    return backbone.lower().replace("-","").replace("_","")

def direct_path_candidates(backbone, dataset, horizon, split):
    root = PATCHTST_DIRECT_ROOT if backbone == "PatchTST" else OTHER_DIRECT_ROOT
    slug = direct_slug(backbone)

    exact = [
        root / f"{dataset}_H{horizon}_{split}_{slug}_direct.npz",
        root / f"{dataset}_H{horizon}_{split}_{backbone}_direct.npz",
        root / f"{dataset}_H{horizon}_{split}_direct.npz",
    ]

    out = [p for p in exact if p.exists()]

    if not out:
        for p in root.rglob("*.npz"):
            s = p.name.lower()
            if dataset.lower() in s and f"h{horizon}" in s and split in s:
                out.append(p)

    return out

def load_direct(backbone, dataset, horizon, split):
    ma,mc = load_manifest(dataset,horizon,split)

    for p in direct_path_candidates(backbone,dataset,horizon,split):
        try:
            with np.load(p,allow_pickle=False) as z:
                if not {"anchor","channel","direct","truth"}.issubset(z.files):
                    continue

                a = np.asarray(z["anchor"],np.int64)
                c = np.asarray(z["channel"],np.int64)
                d = np.asarray(z["direct"],np.float32)
                y = np.asarray(z["truth"],np.float32)

            ok = (
                np.array_equal(a,ma)
                and np.array_equal(c,mc)
                and d.shape == (len(ma),horizon)
                and y.shape == (len(ma),horizon)
                and np.isfinite(d).all()
                and np.isfinite(y).all()
            )

            if ok:
                return {
                    "anchor":a,
                    "channel":c,
                    "direct":d,
                    "truth":y,
                    "path":str(p),
                }
        except Exception:
            continue

    raise FileNotFoundError(
        f"Exact direct cache not found: {backbone} {dataset} H={horizon} {split}"
    )

audit = []
for b in BACKBONES:
    for d in DATASETS:
        for h in HORIZONS:
            ok = True
            for sp in ("val","test"):
                try:
                    load_direct(b,d,h,sp)
                except Exception:
                    ok = False
            audit.append({"backbone":b,"dataset":d,"horizon":h,"ok":ok})

direct_audit = pd.DataFrame(audit)
display(direct_audit.groupby("backbone",as_index=False).agg(total=("ok","size"),ok=("ok","sum")))

assert int(direct_audit.ok.sum()) == 120
print("[PASS] exact Direct val/test cache = 120/120")

,backbone,total,ok
0,DLinear,24,24
1,PatchTST,24,24
2,SegMoE,24,24
3,TimeMixer,24,24
4,iTransformer,24,24


[PASS] exact Direct val/test cache = 120/120


In [4]:
# Cell 4 — raw datasets and exact train-only normalization

EXPECTED_CHANNELS = {
    "ETTh1":7,
    "Weather":21,
    "Electricity":321,
    "Traffic":862,
    "Exchange":8,
    "Solar":137,
}

DATA_CANDIDATES = {
    "ETTh1":[
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETTh1.csv"),
    ],
    "Weather":[
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/dataset/weather.csv"),
    ],
    "Electricity":[
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity.csv"),
    ],
    "Traffic":[
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic.csv"),
    ],
    "Exchange":[
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate.csv"),
    ],
    "Solar":[
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/solar_AL.txt"),
    ],
}

def resolve_data(name):
    p = next((x for x in DATA_CANDIDATES[name] if x.is_file()), None)
    if p is None:
        raise FileNotFoundError(f"{name}: dataset not found")
    return p

def clean_numeric_frame(df):
    return (
        df.apply(pd.to_numeric, errors="coerce")
          .dropna(axis=1, how="all")
          .replace([np.inf,-np.inf],np.nan)
          .interpolate(axis=0,limit_direction="both")
          .ffill().bfill()
    )

def load_dataset(name):
    p = resolve_data(name)

    if name == "Solar":
        df = pd.read_csv(p,header=None)
        if df.shape[1] == 1:
            df = pd.read_csv(p,header=None,sep=r"\s+")
        numeric = clean_numeric_frame(df)
    else:
        df = pd.read_csv(p)
        date_col = next(
            (c for c in df.columns if str(c).lower() in {"date","datetime","timestamp","time"}),
            None,
        )
        val = df.drop(columns=[date_col]) if date_col is not None else df.copy()

        if name in {"Weather","Electricity"} and "OT" in val.columns:
            val = val[[c for c in val.columns if c != "OT"] + ["OT"]]

        numeric = clean_numeric_frame(val)

    raw = numeric.to_numpy(np.float32)
    assert raw.shape[1] == EXPECTED_CHANNELS[name], (name,raw.shape)

    n = len(raw)
    if name == "ETTh1":
        assert raw.shape == (17420,7)
        train_end,val_end = 8640,11520
    else:
        train_end = int(0.70*n)
        val_end = n - int(0.20*n)

    mu = raw[:train_end].mean(axis=0).astype(np.float32)
    sd = raw[:train_end].std(axis=0,ddof=0).astype(np.float32)

    assert np.all(sd > 1e-6), f"{name}: degenerate channel"

    z = ((raw-mu[None,:])/sd[None,:]).astype(np.float32)

    return {
        "path":p,
        "z":z,
        "n":n,
        "channels":raw.shape[1],
        "train_end":train_end,
        "val_end":val_end,
    }

DATA = {d:load_dataset(d) for d in DATASETS}

display(pd.DataFrame([
    {
        "dataset":d,
        "rows":DATA[d]["n"],
        "channels":DATA[d]["channels"],
        "train_end":DATA[d]["train_end"],
        "val_end":DATA[d]["val_end"],
        "path":str(DATA[d]["path"]),
    }
    for d in DATASETS
]))

,dataset,rows,channels,train_end,val_end,path
0,ETTh1,17420,7,8640,11520,/data/Time-Series-Library/dataset/ETT-small/ET...
1,Weather,52696,21,36887,42157,/data/Time-Series-Library/dataset/weather/weat...
2,Electricity,26304,321,18412,21044,/data/Time-Series-Library/dataset/electricity/...
3,Traffic,17544,862,12280,14036,/data/Time-Series-Library/dataset/traffic/traf...
4,Exchange,7588,8,5311,6071,/data/Time-Series-Library/dataset/exchange_rat...
5,Solar,52560,137,36792,42048,/data/Time-Series-Library/dataset/Solar/solar_...


In [5]:
# Cell 5 — Anchored-L2 retrieval forecast generation

def past_windows(z,ch,anchors):
    anchors = np.asarray(anchors,np.int64)
    idx = anchors[:,None] - L + np.arange(L)[None,:]
    return z[idx,int(ch)].astype(np.float32)

def future_residuals(z,ch,anchors,horizon):
    anchors = np.asarray(anchors,np.int64)
    idx = anchors[:,None] + np.arange(int(horizon))[None,:]
    fut = z[idx,int(ch)].astype(np.float32)
    current = z[anchors-1,int(ch)].astype(np.float32)
    return (fut-current[:,None]).astype(np.float32)

def anchored_shape(x):
    x = np.asarray(x,np.float32)
    return (x-x[...,-1:]).astype(np.float32)

def retrieval_cache_path(dataset,horizon,split):
    return RETRIEVAL_CACHE_ROOT / f"{dataset}_H{horizon}_{split}_anchored_l2.npz"

def load_or_build_l2_retrieval(dataset,horizon,split,query_chunk=128):
    qa,qc = load_manifest(dataset,horizon,split)
    p = retrieval_cache_path(dataset,horizon,split)

    if p.exists():
        try:
            with np.load(p,allow_pickle=False) as zc:
                a = np.asarray(zc["anchor"],np.int64)
                c = np.asarray(zc["channel"],np.int64)
                r = np.asarray(zc["retrieval"],np.float32)
                analog = np.asarray(zc["analog_mse"],np.float32)
                rfmse = np.asarray(zc["retrieval_forecast_mse"],np.float32)
                stride = int(np.asarray(zc["memory_stride"]).item())

            if (
                np.array_equal(a,qa)
                and np.array_equal(c,qc)
                and r.shape == (len(qa),horizon)
                and analog.shape == (len(qa),)
                and rfmse.shape == (len(qa),)
                and np.isfinite(r).all()
                and np.isfinite(analog).all()
                and np.isfinite(rfmse).all()
                and stride == MEMORY_STRIDE
            ):
                print("[CACHE]",dataset,horizon,split)
                return {
                    "anchor":a,
                    "channel":c,
                    "retrieval":r,
                    "analog_mse":analog,
                    "retrieval_forecast_mse":rfmse,
                    "path":str(p),
                }
        except Exception:
            pass

    z = DATA[dataset]["z"]
    boundary = DATA[dataset]["train_end"] if split == "val" else DATA[dataset]["val_end"]

    retrieval = np.full((len(qa),horizon),np.nan,np.float32)
    analog = np.full(len(qa),np.nan,np.float32)
    rfmse = np.full(len(qa),np.nan,np.float32)

    # Use a Direct cache only as an exact frozen truth audit.
    ref = load_direct("PatchTST",dataset,horizon,split)
    truth = ref["truth"]

    t0 = time.perf_counter()

    for ch in np.unique(qc):
        rows = np.flatnonzero(qc == ch)

        mem_anchor = np.arange(
            L,
            boundary-horizon+1,
            MEMORY_STRIDE,
            dtype=np.int64,
        )

        assert len(mem_anchor) >= TOP_K
        assert int(mem_anchor.max()) + horizon <= boundary

        mp = past_windows(z,ch,mem_anchor)
        ms = anchored_shape(mp)
        mf = future_residuals(z,ch,mem_anchor,horizon)
        mnorm = np.sum(ms*ms,axis=1)

        for left in range(0,len(rows),query_chunk):
            rr = rows[left:left+query_chunk]
            q_anchor = qa[rr]

            qp = past_windows(z,ch,q_anchor)
            qs = anchored_shape(qp)
            qnorm = np.sum(qs*qs,axis=1,keepdims=True)

            d2 = np.maximum(
                qnorm + mnorm[None,:] - 2.0*(qs @ ms.T),
                0.0,
            )

            top = np.argpartition(d2,TOP_K-1,axis=1)[:,:TOP_K]

            chosen_resid = mf[top]                        # [B,K,H]
            mean_resid = chosen_resid.mean(axis=1)       # [B,H]

            qcur = z[q_anchor-1,int(ch)].astype(np.float32)
            retrieval_abs = mean_resid + qcur[:,None]

            retrieval[rr] = retrieval_abs

            true_abs = truth[rr]
            true_resid = true_abs - qcur[:,None]

            analog[rr] = (
                (chosen_resid-true_resid[:,None,:])**2
            ).mean(axis=(1,2)).astype(np.float32)

            rfmse[rr] = (
                (retrieval_abs-true_abs)**2
            ).mean(axis=1).astype(np.float32)

        print(
            f"  {dataset:11s} H={horizon:3d} {split:4s} "
            f"ch={int(ch):3d} pairs={len(rows):4d} memory={len(mem_anchor):5d}"
        )

    assert np.isfinite(retrieval).all()
    assert np.isfinite(analog).all()
    assert np.isfinite(rfmse).all()

    # Exact scale check against Direct cache truth.
    assert retrieval.shape == truth.shape

    np.savez_compressed(
        p,
        anchor=qa.astype(np.int64),
        channel=qc.astype(np.int16),
        retrieval=retrieval.astype(np.float32),
        analog_mse=analog.astype(np.float32),
        retrieval_forecast_mse=rfmse.astype(np.float32),
        memory_boundary=np.asarray(boundary,np.int64),
        memory_stride=np.asarray(MEMORY_STRIDE,np.int64),
        top_k=np.asarray(TOP_K,np.int64),
        retrieval_name=np.asarray("last_value_anchored_L2_full_memory"),
    )

    print(
        f"[BUILT] {dataset} H={horizon} {split}: "
        f"AFMSE={analog.mean():.6f}, "
        f"RFMSE={rfmse.mean():.6f}, "
        f"{time.perf_counter()-t0:.1f}s"
    )

    return {
        "anchor":qa,
        "channel":qc,
        "retrieval":retrieval,
        "analog_mse":analog,
        "retrieval_forecast_mse":rfmse,
        "path":str(p),
    }

In [6]:
# Cell 6 — build/reuse all 48 Anchored-L2 val/test retrieval caches

retrieval_audit = []

for d in DATASETS:
    for h in HORIZONS:
        for sp in ("val","test"):
            print("\n"+"="*100)
            print(f"Anchored L2 retrieval | {d} | H={h} | {sp}")
            print("="*100)

            x = load_or_build_l2_retrieval(d,h,sp,query_chunk=128)

            retrieval_audit.append({
                "dataset":d,
                "horizon":h,
                "split":sp,
                "pairs":len(x["anchor"]),
                "analog_future_mse":float(x["analog_mse"].mean()),
                "retrieval_forecast_mse":float(x["retrieval_forecast_mse"].mean()),
                "path":x["path"],
            })

            gc.collect()

retrieval_audit = pd.DataFrame(retrieval_audit)
display(retrieval_audit)

assert len(retrieval_audit) == 48
retrieval_audit.to_csv(OUT_ROOT/"anchored_l2_retrieval_audit_48.csv",index=False)

print("[PASS] Anchored-L2 val/test retrieval = 48/48")


Anchored L2 retrieval | ETTh1 | H=96 | val
[CACHE] ETTh1 96 val

Anchored L2 retrieval | ETTh1 | H=96 | test
[CACHE] ETTh1 96 test

Anchored L2 retrieval | ETTh1 | H=192 | val
[CACHE] ETTh1 192 val

Anchored L2 retrieval | ETTh1 | H=192 | test
[CACHE] ETTh1 192 test

Anchored L2 retrieval | ETTh1 | H=336 | val
[CACHE] ETTh1 336 val

Anchored L2 retrieval | ETTh1 | H=336 | test
[CACHE] ETTh1 336 test

Anchored L2 retrieval | ETTh1 | H=720 | val
[CACHE] ETTh1 720 val

Anchored L2 retrieval | ETTh1 | H=720 | test
[CACHE] ETTh1 720 test

Anchored L2 retrieval | Weather | H=96 | val
[CACHE] Weather 96 val

Anchored L2 retrieval | Weather | H=96 | test
[CACHE] Weather 96 test

Anchored L2 retrieval | Weather | H=192 | val
[CACHE] Weather 192 val

Anchored L2 retrieval | Weather | H=192 | test
[CACHE] Weather 192 test

Anchored L2 retrieval | Weather | H=336 | val
[CACHE] Weather 336 val

Anchored L2 retrieval | Weather | H=336 | test
[CACHE] Weather 336 test

Anchored L2 retrieval | Weather

,dataset,horizon,split,pairs,analog_future_mse,retrieval_forecast_mse,path
0,ETTh1,96,val,4096,1.196857,0.887313,/data/code/which_histories_matter_ext/results/...
1,ETTh1,96,test,8192,1.100986,0.737022,/data/code/which_histories_matter_ext/results/...
2,ETTh1,192,val,4096,1.553012,1.185750,/data/code/which_histories_matter_ext/results/...
3,ETTh1,192,test,8192,1.277229,0.828583,/data/code/which_histories_matter_ext/results/...
4,ETTh1,336,val,4096,1.902482,1.486106,/data/code/which_histories_matter_ext/results/...
5,ETTh1,336,test,8192,1.446432,0.896594,/data/code/which_histories_matter_ext/results/...
6,ETTh1,720,val,4096,2.252580,1.757193,/data/code/which_histories_matter_ext/results/...
7,ETTh1,720,test,8192,1.714011,1.041471,/data/code/which_histories_matter_ext/results/...
8,Weather,96,val,4096,1.234972,0.414655,/data/code/which_histories_matter_ext/results/...
9,Weather,96,test,8192,0.839586,0.363352,/data/code/which_histories_matter_ext/results/...


[PASS] Anchored-L2 val/test retrieval = 48/48


In [7]:
# Cell 7 — metric helpers

def mse(a,b):
    a = np.asarray(a,np.float64)
    b = np.asarray(b,np.float64)
    return float(np.mean((a-b)**2))

def mae(a,b):
    a = np.asarray(a,np.float64)
    b = np.asarray(b,np.float64)
    return float(np.mean(np.abs(a-b)))

def combine(direct,retrieval,beta):
    d = np.asarray(direct,np.float32)
    r = np.asarray(retrieval,np.float32)
    return d + float(beta)*(r-d)

def gain_pct(direct_mse,method_mse):
    return 100.0*(direct_mse-method_mse)/direct_mse

def wtl(series,tol=TOL):
    x = pd.Series(series).dropna().to_numpy(float)
    return {
        "W":int((x>tol).sum()),
        "T":int((np.abs(x)<=tol).sum()),
        "L":int((x<-tol).sum()),
    }

print("[PASS] helpers")

[PASS] helpers


In [8]:
# Cell 8 — validation beta curves for all 120 conditions

rows = []

for d in DATASETS:
    for h in HORIZONS:
        l2_val = load_or_build_l2_retrieval(d,h,"val")["retrieval"]

        for b in BACKBONES:
            dv = load_direct(b,d,h,"val")
            direct = dv["direct"]
            truth = dv["truth"]

            dm = mse(direct,truth)

            for beta in BETA_GRID:
                pred = combine(direct,l2_val,beta)
                mm = mse(pred,truth)

                rows.append({
                    "backbone":b,
                    "dataset":d,
                    "horizon":h,
                    "beta":float(beta),
                    "direct_val_mse":dm,
                    "calibrated_val_mse":mm,
                    "mse_ratio_vs_direct":mm/dm,
                    "gain_vs_direct_pct":gain_pct(dm,mm),
                })

val_curve = pd.DataFrame(rows)

assert len(val_curve) == 120*len(BETA_GRID)
assert set(np.round(val_curve.beta.unique(),6)) == set(np.round(BETA_GRID,6))

display(val_curve.head(14))
val_curve.to_csv(OUT_ROOT/"validation_beta_curve_all_conditions.csv",index=False)

print("[PASS] validation curves:",len(val_curve))

[CACHE] ETTh1 96 val
[CACHE] ETTh1 192 val
[CACHE] ETTh1 336 val
[CACHE] ETTh1 720 val
[CACHE] Weather 96 val
[CACHE] Weather 192 val
[CACHE] Weather 336 val
[CACHE] Weather 720 val
[CACHE] Electricity 96 val
[CACHE] Electricity 192 val
[CACHE] Electricity 336 val
[CACHE] Electricity 720 val
[CACHE] Traffic 96 val
[CACHE] Traffic 192 val
[CACHE] Traffic 336 val
[CACHE] Traffic 720 val
[CACHE] Exchange 96 val
[CACHE] Exchange 192 val
[CACHE] Exchange 336 val
[CACHE] Exchange 720 val
[CACHE] Solar 96 val
[CACHE] Solar 192 val
[CACHE] Solar 336 val
[CACHE] Solar 720 val


,backbone,dataset,horizon,beta,direct_val_mse,calibrated_val_mse,mse_ratio_vs_direct,gain_vs_direct_pct
0,PatchTST,ETTh1,96,0.000,0.695547,0.695547,1.000000,0.000000
1,PatchTST,ETTh1,96,0.025,0.695547,0.693528,0.997097,0.290349
2,PatchTST,ETTh1,96,0.050,0.695547,0.691858,0.994695,0.530462
3,PatchTST,ETTh1,96,0.075,0.695547,0.690537,0.992797,0.720338
4,PatchTST,ETTh1,96,0.100,0.695547,0.689566,0.991400,0.859978
5,PatchTST,ETTh1,96,0.150,0.695547,0.688671,0.990115,0.988548
6,PatchTST,ETTh1,96,0.200,0.695547,0.689175,0.990838,0.916171
7,iTransformer,ETTh1,96,0.000,0.697595,0.697595,1.000000,0.000000
8,iTransformer,ETTh1,96,0.025,0.697595,0.696057,0.997796,0.220432
9,iTransformer,ETTh1,96,0.050,0.697595,0.694841,0.996053,0.394692


[PASS] validation curves: 840


In [9]:
# Cell 9 — select one beta per condition using validation only

keys = ["backbone","dataset","horizon"]

selected = (
    val_curve
    .sort_values(
        keys + ["calibrated_val_mse","beta"],
        ascending=[True,True,True,True,True],
    )
    .groupby(keys,as_index=False)
    .first()
    .rename(columns={
        "beta":"selected_beta",
        "calibrated_val_mse":"selected_val_mse",
        "gain_vs_direct_pct":"selected_val_gain_pct",
        "mse_ratio_vs_direct":"selected_val_mse_ratio",
    })
)

assert len(selected) == 120
assert selected.selected_beta.isin(BETA_GRID).all()

beta_dist = (
    selected.groupby("selected_beta",as_index=False)
    .size()
    .rename(columns={"size":"conditions"})
)

display(beta_dist)
display(selected.head(30))

selected.to_csv(OUT_ROOT/"selected_condition_betas.csv",index=False)
beta_dist.to_csv(OUT_ROOT/"selected_beta_distribution.csv",index=False)

print("[PASS] beta selected on validation only: 120/120")

,selected_beta,conditions
0,0.000,14
1,0.025,13
2,0.050,9
3,0.075,6
4,0.100,7
5,0.150,16
6,0.200,55


,backbone,dataset,horizon,selected_beta,direct_val_mse,selected_val_mse,selected_val_mse_ratio,selected_val_gain_pct
0,DLinear,ETTh1,96,0.075,0.671361,0.669693,0.997516,0.248411
1,DLinear,ETTh1,192,0.025,0.885711,0.885372,0.999617,0.038325
2,DLinear,ETTh1,336,0.000,1.102146,1.102146,1.000000,0.000000
3,DLinear,ETTh1,720,0.000,1.237632,1.237632,1.000000,0.000000
4,DLinear,Electricity,96,0.100,0.175206,0.171781,0.980455,1.954468
5,DLinear,Electricity,192,0.075,0.170163,0.168047,0.987568,1.243174
6,DLinear,Electricity,336,0.075,0.183577,0.181620,0.989342,1.065791
7,DLinear,Electricity,720,0.075,0.210886,0.208706,0.989665,1.033484
8,DLinear,Exchange,96,0.200,0.160253,0.149062,0.930168,6.983174
9,DLinear,Exchange,192,0.200,0.298871,0.268167,0.897267,10.273299


[PASS] beta selected on validation only: 120/120


In [10]:
# Cell 10 — frozen TEST evaluation, fixed beta=.1 and condition-calibrated beta

rows = []

for r in selected.itertuples(index=False):
    b = r.backbone
    d = r.dataset
    h = int(r.horizon)
    beta = float(r.selected_beta)

    l2 = load_or_build_l2_retrieval(d,h,"test")["retrieval"]
    dt = load_direct(b,d,h,"test")

    direct = dt["direct"]
    truth = dt["truth"]

    dm = mse(direct,truth)
    dmae = mae(direct,truth)

    fixed_pred = combine(direct,l2,FIXED_BETA)
    fixed_mse = mse(fixed_pred,truth)
    fixed_mae = mae(fixed_pred,truth)

    cal_pred = combine(direct,l2,beta)
    cal_mse = mse(cal_pred,truth)
    cal_mae = mae(cal_pred,truth)

    rows.append({
        "backbone":b,
        "dataset":d,
        "horizon":h,

        "selected_beta":beta,
        "selected_val_gain_pct":float(r.selected_val_gain_pct),

        "direct_mse":dm,
        "fixed01_mse":fixed_mse,
        "condition_cal_mse":cal_mse,

        "direct_mae":dmae,
        "fixed01_mae":fixed_mae,
        "condition_cal_mae":cal_mae,

        "fixed01_gain_vs_direct_pct":gain_pct(dm,fixed_mse),
        "condition_cal_gain_vs_direct_pct":gain_pct(dm,cal_mse),
    })

test120 = pd.DataFrame(rows)

assert len(test120) == 120
assert test120.notna().all().all()

display(test120.head(24))
test120.to_csv(OUT_ROOT/"test_120_condition_calibrated.csv",index=False)

print("[PASS] test evaluation 120/120")

[CACHE] ETTh1 96 test
[CACHE] ETTh1 192 test
[CACHE] ETTh1 336 test
[CACHE] ETTh1 720 test
[CACHE] Electricity 96 test
[CACHE] Electricity 192 test
[CACHE] Electricity 336 test
[CACHE] Electricity 720 test
[CACHE] Exchange 96 test
[CACHE] Exchange 192 test
[CACHE] Exchange 336 test
[CACHE] Exchange 720 test
[CACHE] Solar 96 test
[CACHE] Solar 192 test
[CACHE] Solar 336 test
[CACHE] Solar 720 test
[CACHE] Traffic 96 test
[CACHE] Traffic 192 test
[CACHE] Traffic 336 test
[CACHE] Traffic 720 test
[CACHE] Weather 96 test
[CACHE] Weather 192 test
[CACHE] Weather 336 test
[CACHE] Weather 720 test
[CACHE] ETTh1 96 test
[CACHE] ETTh1 192 test
[CACHE] ETTh1 336 test
[CACHE] ETTh1 720 test
[CACHE] Electricity 96 test
[CACHE] Electricity 192 test
[CACHE] Electricity 336 test
[CACHE] Electricity 720 test
[CACHE] Exchange 96 test
[CACHE] Exchange 192 test
[CACHE] Exchange 336 test
[CACHE] Exchange 720 test
[CACHE] Solar 96 test
[CACHE] Solar 192 test
[CACHE] Solar 336 test
[CACHE] Solar 720 test
[C

,backbone,dataset,horizon,selected_beta,selected_val_gain_pct,direct_mse,fixed01_mse,condition_cal_mse,direct_mae,fixed01_mae,condition_cal_mae,fixed01_gain_vs_direct_pct,condition_cal_gain_vs_direct_pct
0,DLinear,ETTh1,96,0.075,0.248411,0.458847,0.458622,0.458094,0.440262,0.442495,0.441547,0.049097,0.164147
1,DLinear,ETTh1,192,0.025,0.038325,0.526982,0.522879,0.525243,0.487655,0.486392,0.486876,0.778562,0.330093
2,DLinear,ETTh1,336,0.000,0.000000,0.578029,0.571120,0.578029,0.523846,0.519926,0.523846,1.195153,0.000000
3,DLinear,ETTh1,720,0.000,0.000000,0.662874,0.652953,0.662874,0.585476,0.579040,0.585476,1.496647,0.000000
4,DLinear,Electricity,96,0.100,1.954468,0.196789,0.193682,0.193682,0.278960,0.282774,0.282774,1.579246,1.579246
5,DLinear,Electricity,192,0.075,1.243174,0.194258,0.193130,0.192642,0.281145,0.285563,0.283464,0.580191,0.831885
6,DLinear,Electricity,336,0.075,1.065791,0.208630,0.206961,0.206581,0.296362,0.299833,0.297965,0.799810,0.981826
7,DLinear,Electricity,720,0.075,1.033484,0.240387,0.238049,0.237833,0.326563,0.329198,0.327626,0.972390,1.062419
8,DLinear,Exchange,96,0.200,6.983174,0.084852,0.083172,0.082307,0.206948,0.204341,0.202908,1.980564,2.999362
9,DLinear,Exchange,192,0.200,10.273299,0.156909,0.155410,0.155667,0.287624,0.285369,0.284815,0.955452,0.791177


[PASS] test evaluation 120/120


In [11]:
# Cell 11 — overall fixed-vs-calibrated result

fixed_gain = test120["fixed01_gain_vs_direct_pct"]
cal_gain = test120["condition_cal_gain_vs_direct_pct"]

overall = pd.DataFrame([
    {
        "scheme":"Fixed-0.1",
        **wtl(fixed_gain),
        "mean_gain_vs_direct_pct":float(fixed_gain.mean()),
        "median_gain_vs_direct_pct":float(fixed_gain.median()),
        "min_gain_pct":float(fixed_gain.min()),
        "max_gain_pct":float(fixed_gain.max()),
    },
    {
        "scheme":"Condition-Cal",
        **wtl(cal_gain),
        "mean_gain_vs_direct_pct":float(cal_gain.mean()),
        "median_gain_vs_direct_pct":float(cal_gain.median()),
        "min_gain_pct":float(cal_gain.min()),
        "max_gain_pct":float(cal_gain.max()),
    },
])

display(overall)
overall.to_csv(OUT_ROOT/"overall_summary.csv",index=False)

print("Selected beta distribution:")
display(beta_dist)

,scheme,W,T,L,mean_gain_vs_direct_pct,median_gain_vs_direct_pct,min_gain_pct,max_gain_pct
0,Fixed-0.1,81,0,39,1.041473,0.585543,-2.965259,17.249897
1,Condition-Cal,82,14,24,1.854872,0.387997,-4.864517,29.442050


Selected beta distribution:


,selected_beta,conditions
0,0.000,14
1,0.025,13
2,0.050,9
3,0.075,6
4,0.100,7
5,0.150,16
6,0.200,55


In [12]:
# Cell 12 — dataset / backbone / horizon summaries

def grouped_summary(group_cols):
    rows = []

    for key,g in test120.groupby(group_cols,sort=False):
        if not isinstance(key,tuple):
            key = (key,)

        gains = g["condition_cal_gain_vs_direct_pct"]

        row = dict(zip(group_cols,key))
        row.update(wtl(gains))
        row.update({
            "conditions":len(g),
            "mean_gain_vs_direct_pct":float(gains.mean()),
            "median_gain_vs_direct_pct":float(gains.median()),
            "mean_selected_beta":float(g.selected_beta.mean()),
            "beta0_conditions":int((g.selected_beta==0).sum()),
        })
        rows.append(row)

    return pd.DataFrame(rows)

dataset_summary = grouped_summary(["dataset"])
backbone_summary = grouped_summary(["backbone"])
horizon_summary = grouped_summary(["horizon"])

print("DATASET")
display(dataset_summary)

print("BACKBONE")
display(backbone_summary)

print("HORIZON")
display(horizon_summary)

dataset_summary.to_csv(OUT_ROOT/"dataset_summary.csv",index=False)
backbone_summary.to_csv(OUT_ROOT/"backbone_summary.csv",index=False)
horizon_summary.to_csv(OUT_ROOT/"horizon_summary.csv",index=False)

DATASET


,dataset,W,T,L,conditions,mean_gain_vs_direct_pct,median_gain_vs_direct_pct,mean_selected_beta,beta0_conditions
0,ETTh1,7,2,11,20,-0.118196,-0.043299,0.11500,2
1,Electricity,16,4,0,20,0.373026,0.221977,0.03625,4
2,Exchange,14,1,5,20,2.118073,0.860664,0.19000,1
3,Solar,20,0,0,20,6.922414,7.138415,0.19000,0
4,Traffic,10,7,3,20,0.482417,0.019722,0.04500,7
5,Weather,15,0,5,20,1.351495,0.611943,0.19000,0


BACKBONE


,backbone,W,T,L,conditions,mean_gain_vs_direct_pct,median_gain_vs_direct_pct,mean_selected_beta,beta0_conditions
0,DLinear,22,2,0,24,4.706719,2.295593,0.138542,2
1,PatchTST,15,2,7,24,0.356323,0.136510,0.112500,2
2,SegMoE,9,8,7,24,1.590586,0.000000,0.127083,8
3,TimeMixer,15,2,7,24,0.624411,0.138404,0.121875,2
4,iTransformer,21,0,3,24,1.996319,0.574326,0.138542,0


HORIZON


,horizon,W,T,L,conditions,mean_gain_vs_direct_pct,median_gain_vs_direct_pct,mean_selected_beta,beta0_conditions
0,96,21,2,7,30,1.356758,0.352329,0.133333,2
1,192,22,2,6,30,1.547904,0.453583,0.126667,2
2,336,21,5,4,30,2.082986,0.405634,0.122500,5
3,720,18,5,7,30,2.431838,0.272113,0.128333,5


In [13]:
# Cell 13 — calibration generalization audit

audit = test120[
    [
        "backbone","dataset","horizon",
        "selected_beta",
        "selected_val_gain_pct",
        "condition_cal_gain_vs_direct_pct",
    ]
].copy()

audit["val_positive"] = audit.selected_val_gain_pct > 0
audit["test_positive"] = audit.condition_cal_gain_vs_direct_pct > TOL
audit["generalized_sign"] = audit.val_positive == audit.test_positive

corr = float(
    audit.selected_val_gain_pct.corr(
        audit.condition_cal_gain_vs_direct_pct
    )
)

print("Validation/test gain Pearson correlation:",corr)
print("Same improvement sign:",int(audit.generalized_sign.sum()),"/",len(audit))

beta_test = (
    test120.groupby("selected_beta",as_index=False)
    .agg(
        conditions=("dataset","size"),
        mean_val_gain_pct=("selected_val_gain_pct","mean"),
        mean_test_gain_pct=("condition_cal_gain_vs_direct_pct","mean"),
        median_test_gain_pct=("condition_cal_gain_vs_direct_pct","median"),
    )
)

display(beta_test)

audit.to_csv(OUT_ROOT/"calibration_generalization_audit.csv",index=False)
beta_test.to_csv(OUT_ROOT/"beta_specific_test_behavior.csv",index=False)

Validation/test gain Pearson correlation: 0.7166007218964697
Same improvement sign: 96 / 120


,selected_beta,conditions,mean_val_gain_pct,mean_test_gain_pct,median_test_gain_pct
0,0.000,14,0.000000,0.000000,0.000000
1,0.025,13,0.180281,0.141747,0.107384
2,0.050,9,0.535692,0.330093,0.418237
3,0.075,6,0.712094,0.320882,0.498016
4,0.100,7,0.897939,1.088335,1.579246
5,0.150,16,0.947170,0.480846,0.243385
6,0.200,55,4.934732,3.646070,1.813642


In [14]:
# Cell 14 — optional direct comparison with existing calibrated PRR 120-condition result

def discover_file(name_fragment):
    preferred = [
        Path("/data/code/which_histories_matter_ext/results/exp41_condition_calibrated_prr/test_120_condition_calibrated.csv"),
        Path("/data/code/stock_regime_retrieval/strong_forecaster/results/exp41_condition_calibrated_prr/test_120_condition_calibrated.csv"),
    ]

    for p in preferred:
        if p.exists():
            return p

    for base in [Path("/data/code"),Path("/data")]:
        if not base.exists():
            continue
        hits = list(base.rglob(name_fragment))
        for p in hits:
            if "exp41_condition_calibrated_prr" in str(p):
                return p

    return None

PRR_TEST120 = discover_file("test_120_condition_calibrated.csv")

if PRR_TEST120 is not None:
    prr120 = pd.read_csv(PRR_TEST120)

    need = {
        "backbone","dataset","horizon",
        "selected_beta","direct_mse",
        "condition_cal_mse",
        "condition_cal_gain_vs_direct_pct",
    }

    if need.issubset(prr120.columns):
        comp = test120.merge(
            prr120[
                [
                    "backbone","dataset","horizon",
                    "selected_beta","condition_cal_mse",
                    "condition_cal_gain_vs_direct_pct",
                ]
            ],
            on=["backbone","dataset","horizon"],
            suffixes=("_l2","_prr"),
        )

        assert len(comp) == 120

        comp["l2_gain_vs_prr_pct"] = (
            100.0
            * (comp.condition_cal_mse_prr-comp.condition_cal_mse_l2)
            / comp.condition_cal_mse_prr
        )

        print("Calibrated Anchored-L2 vs calibrated PRR")
        print(
            "W/T/L:",
            wtl(comp.l2_gain_vs_prr_pct),
            "| mean:",
            f"{comp.l2_gain_vs_prr_pct.mean():+.3f}%",
            "| median:",
            f"{comp.l2_gain_vs_prr_pct.median():+.3f}%"
        )

        display(
            comp.sort_values("l2_gain_vs_prr_pct",ascending=False).head(20)
        )

        comp.to_csv(
            OUT_ROOT/"calibrated_l2_vs_calibrated_prr_120.csv",
            index=False,
        )
    else:
        print("[INFO] Existing PRR file found but expected columns are missing:",PRR_TEST120)
else:
    print("[INFO] Existing Exp41 calibrated PRR result not found; skipping L2-vs-PRR merge.")

[INFO] Existing PRR file found but expected columns are missing: /data/code/which_histories_matter_ext/results/exp41_condition_calibrated_prr/test_120_condition_calibrated.csv


In [15]:
# Cell 15 — final paper-facing summary (fixed)

fixed_row = overall.loc[overall["scheme"]=="Fixed-0.1"].iloc[0]
cal_row = overall.loc[overall["scheme"]=="Condition-Cal"].iloc[0]

summary = {
    "experiment":"Anchored-L2 downstream 120-condition frozen benchmark",
    "retrieval_training_performed":False,
    "forecaster_training_performed":False,
    "forecaster_inference_performed":False,
    "test_used_for_beta_selection":False,
    "selection_unit":"dataset × horizon × backbone",
    "selection_metric":"validation MSE",
    "beta_grid":BETA_GRID.tolist(),

    "fixed_beta_0p1":{
        "W":int(fixed_row["W"]),
        "T":int(fixed_row["T"]),
        "L":int(fixed_row["L"]),
        "mean_gain_pct":float(fixed_row["mean_gain_vs_direct_pct"]),
        "median_gain_pct":float(fixed_row["median_gain_vs_direct_pct"]),
    },

    "condition_cal":{
        "W":int(cal_row["W"]),
        "T":int(cal_row["T"]),
        "L":int(cal_row["L"]),
        "mean_gain_pct":float(cal_row["mean_gain_vs_direct_pct"]),
        "median_gain_pct":float(cal_row["median_gain_vs_direct_pct"]),
        "beta0_conditions":int((test120["selected_beta"]==0).sum()),
        "beta_positive_conditions":int((test120["selected_beta"]>0).sum()),
    },

    "validation_test_gain_correlation":float(corr),
    "validation_test_sign_agreement":int(audit["generalized_sign"].sum()),
}

(OUT_ROOT/"anchored_l2_downstream_summary.json").write_text(
    json.dumps(summary,indent=2),
    encoding="utf-8",
)

print(json.dumps(summary,indent=2))

print("\n"+"="*100)
print("FINAL RESULT")
print("="*100)

print(
    f"Fixed beta=0.1: "
    f"{int(fixed_row['W'])}W/{int(fixed_row['T'])}T/{int(fixed_row['L'])}L | "
    f"mean={float(fixed_row['mean_gain_vs_direct_pct']):+.3f}% | "
    f"median={float(fixed_row['median_gain_vs_direct_pct']):+.3f}%"
)

print(
    f"Condition-Cal: "
    f"{int(cal_row['W'])}W/{int(cal_row['T'])}T/{int(cal_row['L'])}L | "
    f"mean={float(cal_row['mean_gain_vs_direct_pct']):+.3f}% | "
    f"median={float(cal_row['median_gain_vs_direct_pct']):+.3f}%"
)

print(
    f"beta=0: {int((test120['selected_beta']==0).sum())}/120 | "
    f"val/test corr={float(corr):.3f} | "
    f"sign agreement={int(audit['generalized_sign'].sum())}/120"
)

print("\nSaved under:",OUT_ROOT)


{
  "experiment": "Anchored-L2 downstream 120-condition frozen benchmark",
  "retrieval_training_performed": false,
  "forecaster_training_performed": false,
  "forecaster_inference_performed": false,
  "test_used_for_beta_selection": false,
  "selection_unit": "dataset \u00d7 horizon \u00d7 backbone",
  "selection_metric": "validation MSE",
  "beta_grid": [
    0.0,
    0.025,
    0.05,
    0.075,
    0.1,
    0.15,
    0.2
  ],
  "fixed_beta_0p1": {
    "W": 81,
    "T": 0,
    "L": 39,
    "mean_gain_pct": 1.0414726298668369,
    "median_gain_pct": 0.5855428692406356
  },
  "condition_cal": {
    "W": 82,
    "T": 14,
    "L": 24,
    "mean_gain_pct": 1.8548715007010268,
    "median_gain_pct": 0.3879969882477403,
    "beta0_conditions": 14,
    "beta_positive_conditions": 106
  },
  "validation_test_gain_correlation": 0.7166007218964697,
  "validation_test_sign_agreement": 96
}

FINAL RESULT
Fixed beta=0.1: 81W/0T/39L | mean=+1.041% | median=+0.586%
Condition-Cal: 82W/14T/24L | mean

In [17]:
# Cell 16 — exact paired comparison:
# calibrated Anchored L2 vs calibrated full PRR
# using the already-computed Exp41 test CSV.
#
# No retraining, no retrieval search, no beta re-selection.

from pathlib import Path
import numpy as np
import pandas as pd

PRR_TEST_CSV = (
    PROJECT_ROOT
    / "results"
    / "exp41_condition_calibrated_prr"
    / "test_120_condition_calibrated.csv"
)

assert PRR_TEST_CSV.exists(), PRR_TEST_CSV

prr_raw = pd.read_csv(PRR_TEST_CSV)

print("PRR CSV:", PRR_TEST_CSV)
print("Rows:", len(prr_raw))
print("Columns:")
for c in prr_raw.columns:
    print(" -", c)

assert len(prr_raw) == 120


# ---------------------------------------------------------------------
# 1. Normalize key column names
# ---------------------------------------------------------------------

def find_col(df, candidates, required=True):
    """
    Find a column using:
      1) exact case-insensitive match
      2) normalized-name match
    """
    cols = list(df.columns)

    lower_map = {str(c).lower(): c for c in cols}

    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    def norm(s):
        return (
            str(s).lower()
            .replace("_", "")
            .replace("-", "")
            .replace(" ", "")
            .replace(".", "")
        )

    norm_map = {norm(c): c for c in cols}

    for cand in candidates:
        if norm(cand) in norm_map:
            return norm_map[norm(cand)]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}\n"
            f"Available columns: {cols}"
        )
    return None


backbone_col = find_col(
    prr_raw,
    ["backbone", "model", "forecaster"]
)

dataset_col = find_col(
    prr_raw,
    ["dataset", "data"]
)

horizon_col = find_col(
    prr_raw,
    ["horizon", "H", "pred_len", "prediction_length"]
)


# ---------------------------------------------------------------------
# 2. Find available PRR result columns
# ---------------------------------------------------------------------

direct_mse_col = find_col(
    prr_raw,
    [
        "direct_mse",
        "baseline_mse",
        "direct_test_mse",
    ],
    required=False,
)

prr_cal_mse_col = find_col(
    prr_raw,
    [
        "condition_cal_mse",
        "condition_calibrated_mse",
        "calibrated_mse",
        "prr_cal_mse",
        "prr_condition_cal_mse",
        "test_cal_mse",
    ],
    required=False,
)

prr_gain_col = find_col(
    prr_raw,
    [
        "condition_cal_gain_vs_direct_pct",
        "condition_calibrated_gain_vs_direct_pct",
        "calibrated_gain_vs_direct_pct",
        "prr_gain_vs_direct_pct",
        "gain_vs_direct_pct",
        "test_gain_pct",
    ],
    required=False,
)

selected_beta_col = find_col(
    prr_raw,
    [
        "selected_beta",
        "beta",
        "best_beta",
        "condition_beta",
    ],
    required=False,
)

print("\nDetected schema")
print(" backbone      :", backbone_col)
print(" dataset       :", dataset_col)
print(" horizon       :", horizon_col)
print(" direct MSE    :", direct_mse_col)
print(" PRR cal MSE   :", prr_cal_mse_col)
print(" PRR cal gain  :", prr_gain_col)
print(" selected beta :", selected_beta_col)


# ---------------------------------------------------------------------
# 3. Build a clean PRR 120-condition table
# ---------------------------------------------------------------------

prr = pd.DataFrame({
    "backbone": prr_raw[backbone_col].astype(str),
    "dataset": prr_raw[dataset_col].astype(str),
    "horizon": pd.to_numeric(
        prr_raw[horizon_col], errors="raise"
    ).astype(int),
})

if direct_mse_col is not None:
    prr["prr_direct_mse"] = pd.to_numeric(
        prr_raw[direct_mse_col],
        errors="raise"
    )

if prr_cal_mse_col is not None:
    prr["prr_condition_cal_mse"] = pd.to_numeric(
        prr_raw[prr_cal_mse_col],
        errors="raise"
    )

if prr_gain_col is not None:
    prr["prr_gain_vs_direct_pct"] = pd.to_numeric(
        prr_raw[prr_gain_col],
        errors="raise"
    )

if selected_beta_col is not None:
    prr["prr_selected_beta"] = pd.to_numeric(
        prr_raw[selected_beta_col],
        errors="coerce"
    )


# Normalize backbone spelling.
def normalize_backbone(x):
    s = str(x).strip().lower()
    s = s.replace("-", "").replace("_", "").replace(" ", "")

    mapping = {
        "patchtst": "PatchTST",
        "itransformer": "iTransformer",
        "timemixer": "TimeMixer",
        "segmoe": "SegMoE",
        "dlinear": "DLinear",
    }

    return mapping.get(s, str(x).strip())


prr["backbone"] = prr["backbone"].map(normalize_backbone)
prr["dataset"] = prr["dataset"].str.strip()

test120_cmp = test120.copy()
test120_cmp["backbone"] = (
    test120_cmp["backbone"].map(normalize_backbone)
)
test120_cmp["dataset"] = (
    test120_cmp["dataset"].astype(str).str.strip()
)
test120_cmp["horizon"] = (
    test120_cmp["horizon"].astype(int)
)


# ---------------------------------------------------------------------
# 4. Exact one-to-one merge
# ---------------------------------------------------------------------

KEYS = ["backbone", "dataset", "horizon"]

paired = test120_cmp.merge(
    prr,
    on=KEYS,
    how="inner",
    validate="one_to_one",
)

print("\nMatched rows:", len(paired))

if len(paired) != 120:
    l2_keys = set(
        map(
            tuple,
            test120_cmp[KEYS].itertuples(
                index=False, name=None
            )
        )
    )
    prr_keys = set(
        map(
            tuple,
            prr[KEYS].itertuples(
                index=False, name=None
            )
        )
    )

    print("\nMissing from PRR:")
    print(sorted(l2_keys - prr_keys)[:20])

    print("\nMissing from L2:")
    print(sorted(prr_keys - l2_keys)[:20])

assert len(paired) == 120


# ---------------------------------------------------------------------
# 5. Recover calibrated PRR MSE if CSV stores only gain
# ---------------------------------------------------------------------

# Anchored-L2 notebook's Direct MSE is the benchmark reference.
l2_direct = paired["direct_mse"].to_numpy(float)

if "prr_direct_mse" in paired.columns:
    max_direct_diff = np.max(
        np.abs(
            paired["prr_direct_mse"].to_numpy(float)
            - l2_direct
        )
    )

    print(
        "Max Direct-MSE mismatch "
        "(L2 benchmark vs PRR CSV):",
        max_direct_diff
    )

    # Small serialization differences are acceptable.
    assert max_direct_diff < 5e-6, max_direct_diff


if "prr_condition_cal_mse" not in paired.columns:
    assert (
        "prr_gain_vs_direct_pct" in paired.columns
    ), (
        "PRR CSV contains neither calibrated MSE "
        "nor calibrated gain."
    )

    # gain = 100 * (Direct - PRR) / Direct
    paired["prr_condition_cal_mse"] = (
        paired["direct_mse"]
        * (
            1.0
            - paired["prr_gain_vs_direct_pct"]
              / 100.0
        )
    )


if "prr_gain_vs_direct_pct" not in paired.columns:
    paired["prr_gain_vs_direct_pct"] = (
        100.0
        * (
            paired["direct_mse"]
            - paired["prr_condition_cal_mse"]
        )
        / paired["direct_mse"]
    )


# ---------------------------------------------------------------------
# 6. Exact paired L2-vs-PRR comparison
# ---------------------------------------------------------------------

# Positive => L2 has LOWER MSE than PRR.
paired["l2_gain_vs_prr_pct"] = (
    100.0
    * (
        paired["prr_condition_cal_mse"]
        - paired["condition_cal_mse"]
    )
    / paired["prr_condition_cal_mse"]
)

# Positive => L2 improves more over Direct than PRR,
# measured in percentage points.
paired["gain_difference_pp"] = (
    paired["condition_cal_gain_vs_direct_pct"]
    - paired["prr_gain_vs_direct_pct"]
)


def paired_wtl(x, tol=1e-10):
    x = np.asarray(x, float)

    return {
        "W": int((x > tol).sum()),
        "T": int((np.abs(x) <= tol).sum()),
        "L": int((x < -tol).sum()),
    }


overall_wtl = paired_wtl(
    paired["l2_gain_vs_prr_pct"]
)


# ---------------------------------------------------------------------
# 7. Dataset-level comparison
# ---------------------------------------------------------------------

dataset_rows = []

for d, g in paired.groupby(
    "dataset",
    sort=False
):
    w = paired_wtl(
        g["l2_gain_vs_prr_pct"]
    )

    dataset_rows.append({
        "dataset": d,
        **w,
        "conditions": len(g),

        "L2_mean_gain_vs_Direct_pct":
            float(
                g[
                    "condition_cal_gain_vs_direct_pct"
                ].mean()
            ),

        "PRR_mean_gain_vs_Direct_pct":
            float(
                g[
                    "prr_gain_vs_direct_pct"
                ].mean()
            ),

        "mean_L2_gain_vs_PRR_pct":
            float(
                g[
                    "l2_gain_vs_prr_pct"
                ].mean()
            ),

        "median_L2_gain_vs_PRR_pct":
            float(
                g[
                    "l2_gain_vs_prr_pct"
                ].median()
            ),

        "mean_gain_difference_pp":
            float(
                g[
                    "gain_difference_pp"
                ].mean()
            ),
    })


dataset_pair = pd.DataFrame(
    dataset_rows
)

display(dataset_pair)


# ---------------------------------------------------------------------
# 8. Backbone-level comparison
# ---------------------------------------------------------------------

backbone_rows = []

for b, g in paired.groupby(
    "backbone",
    sort=False
):
    w = paired_wtl(
        g["l2_gain_vs_prr_pct"]
    )

    backbone_rows.append({
        "backbone": b,
        **w,
        "conditions": len(g),

        "L2_mean_gain_vs_Direct_pct":
            float(
                g[
                    "condition_cal_gain_vs_direct_pct"
                ].mean()
            ),

        "PRR_mean_gain_vs_Direct_pct":
            float(
                g[
                    "prr_gain_vs_direct_pct"
                ].mean()
            ),

        "mean_L2_gain_vs_PRR_pct":
            float(
                g[
                    "l2_gain_vs_prr_pct"
                ].mean()
            ),
    })


backbone_pair = pd.DataFrame(
    backbone_rows
)

display(backbone_pair)


# ---------------------------------------------------------------------
# 9. Horizon-level comparison
# ---------------------------------------------------------------------

horizon_rows = []

for h, g in paired.groupby(
    "horizon",
    sort=True
):
    w = paired_wtl(
        g["l2_gain_vs_prr_pct"]
    )

    horizon_rows.append({
        "horizon": int(h),
        **w,
        "conditions": len(g),

        "L2_mean_gain_vs_Direct_pct":
            float(
                g[
                    "condition_cal_gain_vs_direct_pct"
                ].mean()
            ),

        "PRR_mean_gain_vs_Direct_pct":
            float(
                g[
                    "prr_gain_vs_direct_pct"
                ].mean()
            ),

        "mean_L2_gain_vs_PRR_pct":
            float(
                g[
                    "l2_gain_vs_prr_pct"
                ].mean()
            ),
    })


horizon_pair = pd.DataFrame(
    horizon_rows
)

display(horizon_pair)


# ---------------------------------------------------------------------
# 10. Save
# ---------------------------------------------------------------------

paired.to_csv(
    OUT_ROOT
    / "exact_calibrated_l2_vs_prr_120.csv",
    index=False,
)

dataset_pair.to_csv(
    OUT_ROOT
    / "exact_l2_vs_prr_by_dataset.csv",
    index=False,
)

backbone_pair.to_csv(
    OUT_ROOT
    / "exact_l2_vs_prr_by_backbone.csv",
    index=False,
)

horizon_pair.to_csv(
    OUT_ROOT
    / "exact_l2_vs_prr_by_horizon.csv",
    index=False,
)


# ---------------------------------------------------------------------
# 11. Paper-facing summary
# ---------------------------------------------------------------------

print("\n" + "="*100)
print(
    "EXACT CALIBRATED "
    "ANCHORED-L2 vs FULL PRR"
)
print("="*100)

print(
    f"L2 vs PRR: "
    f"{overall_wtl['W']}W/"
    f"{overall_wtl['T']}T/"
    f"{overall_wtl['L']}L"
)

print(
    "Mean L2-vs-PRR MSE gain: "
    f"{paired['l2_gain_vs_prr_pct'].mean():+.3f}%"
)

print(
    "Median L2-vs-PRR MSE gain: "
    f"{paired['l2_gain_vs_prr_pct'].median():+.3f}%"
)

print(
    "\nMean gain vs Direct:"
)

print(
    "  Anchored L2:",
    f"{paired['condition_cal_gain_vs_direct_pct'].mean():+.3f}%"
)

print(
    "  Full PRR   :",
    f"{paired['prr_gain_vs_direct_pct'].mean():+.3f}%"
)

print(
    "  Difference :",
    f"{paired['gain_difference_pp'].mean():+.3f} pp"
)

print(
    "\nSaved to:",
    OUT_ROOT
)

PRR CSV: /data/code/which_histories_matter_ext/results/exp41_condition_calibrated_prr/test_120_condition_calibrated.csv
Rows: 120
Columns:
 - backbone
 - dataset
 - horizon
 - selected_alpha
 - selected_val_gain_pct
 - direct_mse
 - fixed01_mse
 - condition_cal_mse
 - direct_mae
 - fixed01_mae
 - condition_cal_mae
 - fixed01_gain_vs_direct_pct
 - condition_cal_gain_vs_direct_pct

Detected schema
 backbone      : backbone
 dataset       : dataset
 horizon       : horizon
 direct MSE    : direct_mse
 PRR cal MSE   : condition_cal_mse
 PRR cal gain  : condition_cal_gain_vs_direct_pct
 selected beta : None

Matched rows: 120
Max Direct-MSE mismatch (L2 benchmark vs PRR CSV): 1.1102230246251565e-15


,dataset,W,T,L,conditions,L2_mean_gain_vs_Direct_pct,PRR_mean_gain_vs_Direct_pct,mean_L2_gain_vs_PRR_pct,median_L2_gain_vs_PRR_pct,mean_gain_difference_pp
0,ETTh1,16,1,3,20,-0.118196,-0.750437,0.619405,4.823646e-01,0.632242
1,Electricity,7,2,11,20,0.373026,0.397193,-0.024260,-2.108072e-02,-0.024168
2,Exchange,16,1,3,20,2.118073,0.349351,1.853916,1.538634e+00,1.768723
3,Solar,17,0,3,20,6.922414,5.546399,1.421426,1.288149e+00,1.376014
4,Traffic,5,6,9,20,0.482417,0.511756,-0.029567,-1.069463e-13,-0.029339
5,Weather,3,0,17,20,1.351495,2.385351,-1.063192,-1.095224e+00,-1.033856


,backbone,W,T,L,conditions,L2_mean_gain_vs_Direct_pct,PRR_mean_gain_vs_Direct_pct,mean_L2_gain_vs_PRR_pct
0,DLinear,18,1,5,24,4.706719,3.291133,1.549739
1,PatchTST,14,1,9,24,0.356323,-0.074759,0.414839
2,SegMoE,10,6,8,24,1.590586,1.556020,-0.003033
3,TimeMixer,8,2,14,24,0.624411,0.483355,0.141376
4,iTransformer,14,0,10,24,1.996319,1.777262,0.211851


,horizon,W,T,L,conditions,L2_mean_gain_vs_Direct_pct,PRR_mean_gain_vs_Direct_pct,mean_L2_gain_vs_PRR_pct
0,96,20,1,9,30,1.356758,1.135253,0.219311
1,192,15,2,13,30,1.547904,1.219180,0.325313
2,336,15,4,11,30,2.082986,1.593621,0.504808
3,720,14,3,13,30,2.431838,1.678355,0.802386



EXACT CALIBRATED ANCHORED-L2 vs FULL PRR
L2 vs PRR: 64W/10T/46L
Mean L2-vs-PRR MSE gain: +0.463%
Median L2-vs-PRR MSE gain: +0.070%

Mean gain vs Direct:
  Anchored L2: +1.855%
  Full PRR   : +1.407%
  Difference : +0.448 pp

Saved to: /data/code/which_histories_matter_ext/results/anchored_l2_downstream_120
